# Huấn luyện PhoBERT phát hiện tin nhắn lừa đảo (Colab)

Notebook chỉ gọi `scripts/train_phobert.py`; logic nằm trong `scripts/` theo `docs/conventions.md`.

Yêu cầu trước khi chạy:
1. `data/processed/splits/{train,val,test}.csv` đã tạo bằng `scripts/split_dataset.py` và đã commit.
2. Runtime Colab có GPU (Runtime → Change runtime type → GPU).

Kết quả: `scam-detector.onnx`, `vocab.txt`, `bpe.codes`, `tokenizer-golden.json`. Hợp đồng với backend xem README, mục "Hợp đồng giữa notebook huấn luyện và backend".

## 1. Lấy mã nguồn và cài thư viện

Repo private thì đặt token GitHub vào Colab Secrets với tên `GITHUB_TOKEN`.

In [ ]:
import base64
import os

REPOSITORY = "TranSongKy/scambodiadetector"
BRANCH = "main"
CLONE_DIR = "/content/scambodiadetector"

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

auth_header = ""
if token:
    credentials = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth_header = f"-c http.extraheader='Authorization: Basic {credentials}'"

!git {auth_header} clone --quiet --depth 1 --branch {BRANCH} https://github.com/{REPOSITORY}.git {CLONE_DIR}
os.chdir(CLONE_DIR)
!pip install -q -r requirements-train.txt

## 2. Kiểm tra dữ liệu

In [ ]:
!python scripts/validate_dataset.py data/processed/dataset.csv
!wc -l data/processed/splits/*.csv

## 3. Huấn luyện, đánh giá trên tập test và export ONNX

Script tự so logits PyTorch với ONNX và trả lỗi nếu chênh quá `1e-3`.

In [ ]:
OUTPUT_DIR = "/content/model-output"
!python scripts/train_phobert.py data/processed/splits {OUTPUT_DIR} --epochs 4 --batch-size 16 --learning-rate 2e-5

## 4. Tải model về

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/scam-detector-model", "zip", OUTPUT_DIR, ".")
files.download(archive)

## 5. Sau khi tải về (trên máy dev)

```bash
unzip scam-detector-model.zip -d models/
dotnet run scripts/VerifyTokenizer.cs -- models   # tokenizer .NET phải khớp 100% HuggingFace
dotnet run --project src/ScamDetector.Api          # /health phải trả Healthy
```

Ghi dòng kết quả script in ra vào `docs/experiments.md`. Không commit file trong `models/`.